# NER - Nhận diện thực thể có tên (Named Entity Recognition)
## Nhãn: TIME, TASK, LOCATION, PARTNER

## Bước 1: Cài đặt thư viện

In [ ]:
!pip install torch transformers seqeval numpy --break-system-packages -q

## Bước 2: Sinh dữ liệu tổng hợp

In [ ]:
import json
import random

TASKS = [
    "hop du an", "bao cao tien do", "demo san pham", "review hop dong",
    "gap khach hang", "buoi lam viec", "cuoc hop", "thuyet trinh",
    "ky hop dong", "hop trien khai", "an toi", "an trua", "an sang",
    "chao don doi tac", "hop bao", "dao tao nhan vien", "tham quan van phong",
    "phong van ung vien", "hop giao ban", "hop chien luoc",
    "hoc xu ly ngon ngu tu nhien", "hoc machine learning", "hoc lap trinh",
    "on tap", "lam bai tap", "nghien cuu", "doc tai lieu",
]

LOCATIONS = [
    "phong hop tang 3", "phong a", "van phong quan 1", "phong meeting 2",
    "nha hang song viet", "phong du an tang 5", "tru so chinh", "phong hop lon",
    "hoi truong A", "phong giam doc", "tang 7 toa nha diamond", "co so 2",
    "khach san sofitel", "van phong ha noi", "phong hop nho", "hoi truong lon",
    "phong vip tang 10", "nha hang pho bien", "co so thu duc", "phong tu van",
    # Online platforms
    "teams", "zoom", "google meet", "skype", "webex",
    "microsoft teams", "google classroom", "zalo", "discord",
]

PARTNERS = [
    "g-solutions", "cong ty an binh", "seatech", "blue ocean",
    "sunrise holdings", "acme", "tap doan minh phat", "cong ty tech viet",
    "viettel", "fpt software", "cong ty abc", "doanh nghiep xyz",
    "nhom doi tac nhat ban", "khach hang han quoc", "cong ty hai dang",
    "green tech", "smart solutions", "cong ty alpha", "doi tac chau au",
    "cong ty beta group"
]

# Mẫu thời gian — sẽ được gán nhãn TIME
TIME_LIST = [
    "sang mai luc {h}h{m}", "chieu mai luc {h}h{m}", "toi nay luc {h}h{m}",
    "sang thu {d} luc {h}h{m}", "ngay {dd}/{mm} luc {h}h{m}",
    "thu {d} luc {h}h{m}", "mai luc {h}h{m}", "{h}h{m} sang mai",
    "sang thu {d}", "thu {d} tuan sau luc {h}h{m}", "ngay {dd}/{mm}",
    "{h}h", "{h}h{m}", "luc {h}h",
]

TEMPLATES = [
    "{time} {task} tai {location} voi {partner}",
    "{time} {task} voi {partner} tai {location}",
    "{time} gap {partner} de {task} tai {location}",
    "{time} co {task} tai {location} voi {partner}",
    "{time} co buoi {task} voi {partner} o {location}",
    "{time} {partner} den {location} de {task}",
    "{time} {task} tren {location}",
    "{time} {task} tren {location} voi {partner}",
]

def random_time():
    h = random.randint(7, 21)
    m = random.choice(["", "00", "30", "15", "45"])
    d = random.randint(2, 7)
    dd = random.randint(1, 28)
    mm = random.randint(1, 12)
    return random.choice(TIME_LIST).format(h=h, m=m, d=d, dd=dd, mm=mm)

def find_span(text, substring):
    idx = text.find(substring)
    return (idx, idx + len(substring)) if idx != -1 else (None, None)

def build_sample():
    task     = random.choice(TASKS)
    location = random.choice(LOCATIONS)
    partner  = random.choice(PARTNERS)
    time_str = random_time()

    template = random.choice(TEMPLATES)
    # Bỏ partner nếu template không có {partner}
    try:
        text = template.format(time=time_str, task=task, location=location, partner=partner)
    except KeyError:
        return None

    entities = []
    # TIME
    s, e = find_span(text, time_str)
    if s is not None:
        entities.append({"start": s, "end": e, "label": "TIME"})
    # TASK
    s, e = find_span(text, task)
    if s is not None:
        entities.append({"start": s, "end": e, "label": "TASK"})
    # LOCATION
    s, e = find_span(text, location)
    if s is not None:
        entities.append({"start": s, "end": e, "label": "LOCATION"})
    # PARTNER (chỉ thêm nếu có trong text)
    s, e = find_span(text, partner)
    if s is not None:
        entities.append({"start": s, "end": e, "label": "PARTNER"})

    return {"text": text, "entities": entities} if entities else None

def generate(n=1000):
    samples, seen = [], set()
    attempts = 0
    while len(samples) < n and attempts < n * 10:
        attempts += 1
        s = build_sample()
        if s and s["text"] not in seen:
            seen.add(s["text"])
            samples.append(s)
    return samples

random.seed(42)

# Load dữ liệu gốc
existing = []
with open("data/train.jsonl") as f:
    for line in f:
        line = line.strip()
        if line:
            existing.append(json.loads(line))

# Sinh thêm
generated = generate(1000)
all_data = existing + generated
random.shuffle(all_data)

# Chia train/test 80/20
split = int(len(all_data) * 0.8)
train_data = all_data[:split]
test_data  = all_data[split:]

# Lưu file
with open("data/train_full.jsonl", "w") as f:
    for s in train_data:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

with open("data/test.jsonl", "w") as f:
    for s in test_data:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"Train: {len(train_data)} mẫu")
print(f"Test:  {len(test_data)} mẫu")
print(f"\nVí dụ mẫu:")
for s in train_data[:3]:
    print(f"\n  text: {s['text']}")
    for e in sorted(s["entities"], key=lambda x: x["start"]):
        print(f"  [{e['label']}] '{s['text'][e['start']:e['end']]}'")


## Bước 3: Định nghĩa nhãn và Dataset

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN    = 128

LABELS   = ["O", "B-TIME", "I-TIME", "B-TASK", "I-TASK", "B-LOCATION", "I-LOCATION", "B-PARTNER", "I-PARTNER"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

def char_to_bio(text, entities):
    tokens = text.split()
    char_labels = ["O"] * len(text)
    for ent in entities:
        for i in range(ent["start"], ent["end"]):
            if i < len(char_labels):
                char_labels[i] = ("B-" if i == ent["start"] else "I-") + ent["label"]
    word_labels, pos = [], 0
    for token in tokens:
        word_labels.append(char_labels[pos] if pos < len(char_labels) else "O")
        pos += len(token) + 1
    return tokens, word_labels

class NERDataset(Dataset):
    def __init__(self, samples, tokenizer):
        self.data = []
        for sample in samples:
            tokens, word_labels = char_to_bio(sample["text"], sample["entities"])
            enc = tokenizer(tokens, is_split_into_words=True, truncation=True,
                            max_length=MAX_LEN, padding="max_length", return_tensors="pt")
            word_ids = enc.word_ids()
            label_ids, prev = [], None
            for wid in word_ids:
                if wid is None:
                    label_ids.append(-100)
                elif wid != prev:
                    label_ids.append(LABEL2ID.get(word_labels[wid], 0))
                else:
                    tag = word_labels[wid]
                    if tag.startswith("B-"): tag = "I-" + tag[2:]
                    label_ids.append(LABEL2ID.get(tag, 0))
                prev = wid
            self.data.append({
                "input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels":         torch.tensor(label_ids, dtype=torch.long),
            })

    def __len__(self):  return len(self.data)
    def __getitem__(self, i): return self.data[i]

print("Đang tải tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = NERDataset(train_data, tokenizer)
test_dataset  = NERDataset(test_data,  tokenizer)
print(f"Dataset sẵn sàng — Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print(f"Nhãn: {LABELS}")


## Bước 4: Huấn luyện mô hình

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
)
from seqeval.metrics import classification_report, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_labels, pred_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        tr, pr = [], []
        for p, l in zip(pred_seq, label_seq):
            if l == -100: continue
            tr.append(ID2LABEL[l])
            pr.append(ID2LABEL[p])
        true_labels.append(tr)
        pred_labels.append(pr)
    return {"f1": f1_score(true_labels, pred_labels)}

print("Đang tải model...")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

args = TrainingArguments(
    output_dir="model_output",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Bắt đầu huấn luyện...")
trainer.train()


## Bước 5: Đánh giá kết quả

In [ ]:
results = trainer.evaluate()
print(f"Test F1: {results['eval_f1']:.4f}")

# Báo cáo chi tiết
pred_out = trainer.predict(test_dataset)
preds    = np.argmax(pred_out.predictions, axis=-1)
labels   = pred_out.label_ids

true_labels, pred_labels = [], []
for pred_seq, label_seq in zip(preds, labels):
    tr, pr = [], []
    for p, l in zip(pred_seq, label_seq):
        if l == -100: continue
        tr.append(ID2LABEL[l])
        pr.append(ID2LABEL[p])
    true_labels.append(tr)
    pred_labels.append(pr)

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels))

# Lưu model
trainer.save_model("model_output")
tokenizer.save_pretrained("model_output")
print("Model đã được lưu vào model_output/")


## Bước 6: Dự đoán văn bản mới

In [ ]:
def predict(text: str) -> list:
    tokens = text.split()
    enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                    truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**enc)
    preds    = torch.argmax(outputs.logits[0], dim=-1).tolist()
    word_ids = enc.word_ids()

    # Token → word
    word_preds, prev = [], None
    for pid, wid in zip(preds, word_ids):
        if wid is None or wid == prev: 
            prev = wid
            continue
        word_preds.append((tokens[wid], ID2LABEL[pid]))
        prev = wid

    # Gom BIO → entity
    entities, current = [], None
    for word, label in word_preds:
        if label.startswith("B-"):
            if current: entities.append(current)
            current = {"text": word, "label": label[2:]}
        elif label.startswith("I-") and current:
            current["text"] += " " + word
        else:
            if current: entities.append(current)
            current = None
    if current: entities.append(current)
    return entities

# Test thử
test_texts = [
    "sang mai luc 9h hop du an voi cong ty fpt tai phong hop tang 5",
    "chieu thu 4 ky hop dong voi green tech tai van phong quan 3",
    "toi nay 19h an toi voi doi tac viettel tai nha hang sen viet",
]

for text in test_texts:
    print(f"Text: {text}")
    for ent in predict(text):
        print(f"  [{ent['label']}] {ent['text']}")
    print()


## Bước 7: Nhập văn bản và xem kết quả

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

text_input = widgets.Textarea(
    placeholder="Nhập đoạn văn bản tiếng Việt vào đây...",
    layout=widgets.Layout(width="100%", height="80px"),
)

btn = widgets.Button(
    description="Phân tích",
    button_style="primary",
    layout=widgets.Layout(width="120px"),
)

output = widgets.Output()

COLORS = {
    "TIME":     "#9C27B0",
    "TASK":     "#4CAF50",
    "LOCATION": "#2196F3",
    "PARTNER":  "#FF9800",
}

def on_click(b):
    with output:
        clear_output()
        text = text_input.value.strip()
        if not text:
            print("Vui lòng nhập văn bản!")
            return

        entities = predict(text)

        if not entities:
            print("Không tìm thấy thực thể nào.")
            return

        spans = []
        for ent in entities:
            idx = text.find(ent["text"])
            if idx != -1:
                spans.append((idx, idx + len(ent["text"]), ent["label"], ent["text"]))
        spans.sort(key=lambda x: x[0])

        html = ""
        prev = 0
        for start, end, label, ent_text in spans:
            html += text[prev:start]
            color = COLORS.get(label, "#9E9E9E")
            html += (
                f'<mark style="background:{color};color:white;padding:2px 6px;'
                f'border-radius:4px;margin:0 2px;">'
                f'{ent_text} <sup style="font-size:0.7em">{label}</sup></mark>'
            )
            prev = end
        html += text[prev:]

        display(HTML(f"<div style='font-size:16px;line-height:2'>{html}</div>"))

        rows = "".join(
            f"<tr><td style='padding:6px 12px'><b style='color:{COLORS.get(e['label'], '#000')}'>{e['label']}</b></td>"
            f"<td style='padding:6px 12px'>{e['text']}</td></tr>"
            for e in entities
        )
        display(HTML(
            f"<table border='1' cellspacing='0' style='margin-top:12px;border-collapse:collapse'>"
            f"<tr style='background:#f0f0f0'><th style='padding:6px 12px'>Nhãn</th>"
            f"<th style='padding:6px 12px'>Thực thể</th></tr>{rows}</table>"
        ))

btn.on_click(on_click)
display(text_input, btn, output)
